# Data Preprocessing Pipeline
**CS/DS 3262 Final Project — TikTok Binge-Session Classifier**

This notebook documents every preprocessing decision made before model training:
1. Raw feature extraction from the TikTok export
2. Missing-value audit and handling
3. Categorical encoding (cyclical + one-hot)
4. Numeric scaling
5. Correlation heatmap and feature selection

## 1. Imports & Data Loading

In [ ]:
import json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

ZIP_PATH = Path('TikTok_Data_1776821537.zip')
DATE_FMT = '%Y-%m-%d %H:%M:%S'

def parse_date(s):
    try: return datetime.strptime((s or '').strip(), DATE_FMT)
    except: return None

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open('user_data_tiktok.json') as f:
        data = json.load(f)

act = data['Your Activity']
laf = data['Likes and Favorites']
print('Export loaded.')

## 2. Raw Event Extraction

Seven event types are extracted from the export and tagged with a **primitive class** (`attention`, `intent`, `social`, `preference`). All events without a parseable timestamp are discarded — the downstream session model requires a temporal ordering.

In [ ]:
rows = []
def add(source, primitive, records, date_key='Date'):
    for r in records:
        ts = parse_date(r.get(date_key) or r.get('date', ''))
        if ts:
            rows.append({'ts': ts, 'source': source, 'primitive': primitive})

add('watch',          'attention',  act.get('Watch History',  {}).get('VideoList', []))
add('search',         'intent',     act.get('Searches',       {}).get('SearchList', []))
add('share',          'social',     act.get('Share History',  {}).get('ShareHistoryList', []))
add('repost',         'social',     act.get('Reposts',        {}).get('RepostList', []))
add('comment',        'social',     data.get('Comment', {}).get('Comments', {}).get('CommentsList', []))
add('like',           'preference', laf.get('Likes', {}).get('ItemFavoriteList', []), date_key='date')
add('favorite_video', 'preference', laf.get('Favorite Videos', {}).get('FavoriteVideoList', []))
add('favorite_sound', 'preference', laf.get('Favorite Sounds', {}).get('FavoriteSoundList', []))

events = pd.DataFrame(rows).sort_values('ts').reset_index(drop=True)
print(f'Total timestamped events: {len(events):,}')
events['source'].value_counts()

## 3. Session Segmentation

Events are grouped into sessions using a **30-minute inactivity gap** — the standard threshold used throughout the existing TypeScript analysis codebase. Any two consecutive events separated by >= 30 minutes are assigned to different sessions.

In [ ]:
SESSION_GAP = timedelta(minutes=30)

sids = [0]
for i in range(1, len(events)):
    sids.append(sids[-1] + (1 if events.loc[i,'ts'] - events.loc[i-1,'ts'] > SESSION_GAP else 0))
events['session_id'] = sids

print(f'Total sessions: {events["session_id"].nunique():,}')

## 4. Session-Level Feature Extraction

Each session is collapsed into a single feature row. The 12 raw features are:

| Feature | Type | Description |
|---|---|---|
| `event_count` | int | Total events in session |
| `duration_min` | float | Session span in minutes |
| `peak_epm` | float | Events per minute |
| `watch_share` | float | Fraction of watch events |
| `search_share` | float | Fraction of search events |
| `social_share` | float | Fraction of social events |
| `pref_share` | float | Fraction of preference events |
| `cascade_count` | int | Search→watch transitions |
| `hour_of_day` | int | Session start hour (0-23) |
| `day_of_week` | int | Session start day (0=Mon, 6=Sun) |
| `has_search` | binary int | 1 if session contains a search |
| `has_social` | binary int | 1 if session contains a social event |

In [ ]:
def extract_features(grp):
    n = len(grp)
    duration = (grp['ts'].max() - grp['ts'].min()).total_seconds() / 60.0
    start = grp['ts'].min()
    cascade, last_search = 0, False
    for src in grp.sort_values('ts')['source']:
        if src == 'search': last_search = True
        elif src == 'watch' and last_search: cascade += 1; last_search = False
    return {
        'session_id':    grp['session_id'].iloc[0],
        'date':          start.date(),
        'event_count':   n,
        'duration_min':  round(duration, 4),
        'peak_epm':      round(n / max(duration, 1.0), 4),
        'watch_share':   round((grp['primitive']=='attention').mean(), 4),
        'search_share':  round((grp['source']=='search').mean(), 4),
        'social_share':  round((grp['primitive']=='social').mean(), 4),
        'pref_share':    round((grp['primitive']=='preference').mean(), 4),
        'cascade_count': cascade,
        'hour_of_day':   start.hour,
        'day_of_week':   start.weekday(),
        'has_search':    int((grp['source']=='search').any()),
        'has_social':    int((grp['primitive']=='social').any()),
    }

sessions = pd.DataFrame([extract_features(g) for _, g in events.groupby('session_id')])

daily_counts    = events.groupby(events['ts'].dt.date).size()
top10_threshold = daily_counts.quantile(0.90)
median_daily    = daily_counts.median()
binge_days      = set(daily_counts[(daily_counts >= top10_threshold) & (daily_counts >= 2*median_daily)].index)
sessions['binge'] = sessions['date'].apply(lambda d: int(d in binge_days))

FEATURE_COLS = ['event_count','duration_min','peak_epm','watch_share','search_share',
                'social_share','pref_share','cascade_count','hour_of_day','day_of_week',
                'has_search','has_social']

df_raw = sessions[FEATURE_COLS + ['binge']].copy()
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

## 5. Missing-Value Audit

Each column is inspected for nulls. The audit also checks for implicit missing values — zeros that may represent "no data" rather than a true zero count (e.g., `duration_min = 0` for single-event sessions is genuine, not missing).

In [ ]:
null_counts = df_raw.isnull().sum()
zero_counts = (df_raw[FEATURE_COLS] == 0).sum()

audit = pd.DataFrame({
    'dtype':      df_raw[FEATURE_COLS].dtypes,
    'null_count': null_counts[FEATURE_COLS],
    'null_%':     (null_counts[FEATURE_COLS] / len(df_raw) * 100).round(2),
    'zero_count': zero_counts,
    'zero_%':     (zero_counts / len(df_raw) * 100).round(1),
    'handling': [
        'No nulls — zeros are genuine single-event sessions',
        'No nulls — zero means session started and ended in same second',
        'No nulls — floored at n/max(duration,1) to avoid div/0',
        'No nulls — 0.0 means no watch events in session',
        'No nulls — 0.0 means no search events in session',
        'No nulls — 0.0 means no social events in session',
        'No nulls — 0.0 means no preference events in session',
        'No nulls — 0 means no search to watch transitions',
        'No nulls — derived from timestamp, always 0-23',
        'No nulls — derived from timestamp, always 0-6',
        'No nulls — binary derived from search_share',
        'No nulls — binary derived from social_share',
    ]
})

print('Missing-value audit:')
audit

**Finding:** There are zero null values in any column. This is by construction: every session row is derived from at least one timestamped event, so `hour_of_day` and `day_of_week` are always defined; share features are computed as fractions over a known denominator; and `peak_epm` is floored at 1 minute to prevent division-by-zero. No imputation is required.

The high zero-rate on `watch_share` (72%), `search_share` (97%), and `cascade_count` (88%) reflects genuine sparsity: most short sessions consist entirely of likes/favorites with no video watching or searching. These zeros carry signal and are retained as-is.

## 6. Categorical Encoding

Two features are conceptually categorical but stored as integers:

- **`hour_of_day`** (0-23): Time is circular — hour 23 is adjacent to hour 0, but a raw integer encoding treats them as maximally distant. **Cyclical encoding** using sine/cosine projections preserves the circular topology.
- **`day_of_week`** (0-6): Same circular argument — Sunday (6) is adjacent to Monday (0).

This produces 2 columns per feature instead of 7 or 24, avoiding high dimensionality while preserving temporal proximity that one-hot encoding destroys.

In [ ]:
df = df_raw.copy()

df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)

df.drop(columns=['hour_of_day', 'day_of_week'], inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sample = df.sample(500, random_state=42)
axes[0].scatter(sample['hour_sin'], sample['hour_cos'],
                c=df_raw.loc[sample.index, 'hour_of_day'], cmap='twilight', s=15, alpha=0.7)
axes[0].set_title('Hour of Day — Cyclical Encoding')
axes[0].set_xlabel('sin(hour x 2pi/24)')
axes[0].set_ylabel('cos(hour x 2pi/24)')
axes[0].set_aspect('equal')

axes[1].scatter(sample['dow_sin'], sample['dow_cos'],
                c=df_raw.loc[sample.index, 'day_of_week'], cmap='tab10', s=15, alpha=0.7)
axes[1].set_title('Day of Week — Cyclical Encoding')
axes[1].set_xlabel('sin(dow x 2pi/7)')
axes[1].set_ylabel('cos(dow x 2pi/7)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('plot_cyclical_encoding.png', dpi=150)
plt.show()
print('Encoded columns:', ['hour_sin','hour_cos','dow_sin','dow_cos'])

## 7. Numeric Scaling

**Choice: StandardScaler** (zero mean, unit variance) applied to all continuous features.

- `event_count` has extreme right-skew (max=1,226, 75th pct=5) — MinMaxScaler would compress 99% of values into a tiny range. StandardScaler handles this better.
- Binary features (`has_search`, `has_social`) and cyclical features (already in [-1, 1]) are **excluded** from scaling.
- The scaler is fit on training data only inside each CV fold in `ml_pipeline.py`. Here it is fit on the full dataset for inspection only.

In [ ]:
SCALE_COLS  = ['event_count','duration_min','peak_epm',
               'watch_share','search_share','social_share','pref_share','cascade_count']
PASSTHROUGH = ['has_search','has_social','hour_sin','hour_cos','dow_sin','dow_cos']

scaler = StandardScaler()
df_scaled_vals = scaler.fit_transform(df[SCALE_COLS])
df_scaled = pd.DataFrame(df_scaled_vals, columns=SCALE_COLS, index=df.index)
df_scaled[PASSTHROUGH] = df[PASSTHROUGH]
df_scaled['binge'] = df['binge']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df_raw['event_count'].clip(upper=100), bins=50, color='#95a5a6')
axes[0].set_title('event_count — Raw (clipped at 100)')
axes[0].set_xlabel('Value')
axes[1].hist(df_scaled['event_count'], bins=50, color='#3498db')
axes[1].set_title('event_count — After StandardScaler')
axes[1].set_xlabel('Standard deviations from mean')
plt.tight_layout()
plt.savefig('plot_scaling_comparison.png', dpi=150)
plt.show()

print('Scaled feature stats:')
df_scaled[SCALE_COLS].describe().round(3)

## 8. Feature Correlation Heatmap

A Pearson correlation matrix is computed on the preprocessed feature set. Scaling does not change pairwise correlations, so the heatmap is equivalent for the raw set.

In [ ]:
DISPLAY_NAMES = {
    'event_count':   'Event Count',   'duration_min':  'Duration (min)',
    'peak_epm':      'Peak EPM',      'watch_share':   'Watch Share',
    'search_share':  'Search Share',  'social_share':  'Social Share',
    'pref_share':    'Pref Share',    'cascade_count': 'Cascade Count',
    'has_search':    'Has Search',    'has_social':    'Has Social',
    'hour_sin':      'Hour sin',      'hour_cos':      'Hour cos',
    'dow_sin':       'DoW sin',       'dow_cos':       'DoW cos',
    'binge':         'Binge (target)',
}

corr_cols  = SCALE_COLS + PASSTHROUGH + ['binge']
corr_df    = df_scaled[corr_cols].rename(columns=DISPLAY_NAMES)
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 8},
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax
)
ax.set_title('Feature Correlation Matrix (Pearson)', pad=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', dpi=150)
plt.show()

## 9. Correlation Analysis & Feature Selection

In [ ]:
target_corr = corr_matrix['Binge (target)'].drop('Binge (target)').sort_values(key=abs, ascending=False)
print('Correlations with binge label (|r| descending):')
print(target_corr.round(3).to_string())

print('\nFeature pairs with |r| > 0.80 (multicollinearity risk):')
found = False
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.80 and corr_matrix.columns[j] != 'Binge (target)':
            print(f'  {corr_matrix.columns[i]} x {corr_matrix.columns[j]} : r = {r:.3f}')
            found = True
if not found:
    print('  None found.')

## 10. Target Correlation Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in target_corr.values]
ax.barh(target_corr.index[::-1], target_corr.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson r with binge label')
ax.set_title('Feature-Target Correlations')
for i, val in enumerate(target_corr.values[::-1]):
    ax.text(val + (0.005 if val >= 0 else -0.005), i, f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.savefig('plot_target_correlations.png', dpi=150)
plt.show()

## 11. Preprocessing Decision Summary

In [ ]:
decisions = pd.DataFrame([
    ('event_count',   'int',   'None (no nulls)',       'StandardScaler',     'Retained'),
    ('duration_min',  'float', 'None (no nulls)',       'StandardScaler',     'Retained'),
    ('peak_epm',      'float', 'None — div/0 floored',  'StandardScaler',     'Retained'),
    ('watch_share',   'float', 'None (no nulls)',       'StandardScaler',     'Retained'),
    ('search_share',  'float', 'None (no nulls)',       'StandardScaler',     'Retained'),
    ('social_share',  'float', 'None (no nulls)',       'StandardScaler',     'Retained'),
    ('pref_share',    'float', 'None (no nulls)',       'StandardScaler',     'Retained'),
    ('cascade_count', 'int',   'None (no nulls)',       'StandardScaler',     'Retained'),
    ('hour_of_day',   'int',   'None (no nulls)',       'Cyclical (sin/cos)', 'Replaced by hour_sin, hour_cos'),
    ('day_of_week',   'int',   'None (no nulls)',       'Cyclical (sin/cos)', 'Replaced by dow_sin, dow_cos'),
    ('has_search',    'binary','None (no nulls)',       'None (already 0/1)', 'Retained'),
    ('has_social',    'binary','None (no nulls)',       'None (already 0/1)', 'Retained'),
], columns=['Feature','Dtype','Missing Handling','Encoding/Scaling','Decision'])
decisions

In [ ]:
FINAL_FEATURES = SCALE_COLS + PASSTHROUGH
X_final = df_scaled[FINAL_FEATURES].values
y_final = df_scaled['binge'].values

print(f'Final feature matrix : {X_final.shape}')
print(f'Binge sessions       : {y_final.sum()} ({y_final.mean():.1%})')
print(f'Normal sessions      : {len(y_final)-y_final.sum()} ({1-y_final.mean():.1%})')

df_scaled[FINAL_FEATURES + ['binge']].to_csv('sessions_preprocessed.csv', index=False)
print('Saved: sessions_preprocessed.csv')